# ViSNN walkthrough — T=1 ANN→SNN conversion, both tracks

Runs the full pipeline interactively on synthetic data (no dataset download).
Swap `SYNTHETIC = False` and set the roots once KITTI / COCO are in place.

**Phase 1** data → **Phase 2** calibrate/convert/freeze → **Phase 3** train → **Phase 4** validate.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))

import matplotlib.pyplot as plt
import torch

import config, utils

SYNTHETIC = True
utils.set_seed(config.SEED)
config.ensure_dirs()
device = utils.get_device()
print('device:', device, '| torch', torch.__version__)

## Phase 1 — Data pipeline (Role 2)

The one invariant that matters: the depth target receives *exactly* the same
spatial transform as the RGB frame. Zero pixels in the target mean "no LiDAR
return", not "zero metres".

In [ ]:
from data.loaders import build_depth_loaders
from losses.depth_loss import valid_mask

train_loader, val_loader = build_depth_loaders(
    root=config.KITTI_ROOT, synthetic=SYNTHETIC, batch_size=8,
    num_workers=0, device=device, max_train=128, max_val=32)

images, depths = next(iter(train_loader))
print('images', tuple(images.shape), '| depths', tuple(depths.shape))
print('valid ground-truth pixels: '
      f'{valid_mask(depths).float().mean().item():.1%}')

In [ ]:
from data.transforms import denormalize

fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for i in range(3):
    axes[0][i].imshow(denormalize(images[i]).permute(1, 2, 0).numpy())
    axes[0][i].set_title('RGB'); axes[0][i].axis('off')
    masked = depths[i, 0].clone(); masked[~valid_mask(depths[i, 0])] = float('nan')
    axes[1][i].imshow(masked.numpy(), cmap='magma')
    axes[1][i].set_title('depth (m)'); axes[1][i].axis('off')
plt.tight_layout(); plt.show()

## Phase 2 — Calibration and neuromorphic surgery (Role 1)

Profile the *continuous* backbone, take the per-channel top-p percentile with a
2px border excluded, then swap every ReLU/ReLU6 for a `StrictT1SFN` and freeze.

In [ ]:
from pipeline import build_depth_pipeline

depth_model, thresholds = build_depth_pipeline(
    train_loader, device, backbone_name='mobilenet_v2', pretrained=True,
    calibration_batches=5, percentile=config.TOP_P,
    crop_margin=config.CROP_MARGIN, lambda_=1.0, fire_fn='binary', timesteps=1)

In [ ]:
# Threshold distribution for one layer — this is what the SNN fires against.
layer = list(thresholds)[6]
plt.figure(figsize=(7, 3.5))
plt.hist(thresholds[layer], bins=40, color='#4C78A8')
plt.xlabel('theta (per channel)'); plt.ylabel('channels')
plt.title(f'Calibrated thresholds — {layer}')
plt.tight_layout(); plt.show()

In [ ]:
from models.snn import set_spike_tracking, spike_report

# Encoder output is strictly {0, theta_c} per channel — verify it.
set_spike_tracking(depth_model.encoder, True, True)
with torch.no_grad():
    features = depth_model.encoder(images.to(device))
report = spike_report(depth_model.encoder)
set_spike_tracking(depth_model.encoder, False)

print('features', tuple(features.shape))
print(f'overall spike rate: {report["overall"]:.3f}')
print('distinct values in channel 0:',
      torch.unique(features[:, 0]).tolist()[:5])

## Phase 3 — Train the continuous decoder (Role 1)

AdamW touches the decoder only. The frozen spiking encoder runs under
`no_grad`, so the graph begins at `decode()`.

In [ ]:
from losses.depth_loss import DepthLoss
from validation.metrics_depth import evaluate_depth, format_metrics

optimizer = torch.optim.AdamW(depth_model.head.parameters(), lr=1e-3,
                              weight_decay=config.WEIGHT_DECAY)
criterion = DepthLoss(alpha=config.GRAD_LOSS_ALPHA)
history = {'loss': [], 'val_rmse': []}

for epoch in range(1, 6):
    depth_model.train()
    running = utils.AverageMeter()
    for batch_images, batch_depths in train_loader:
        batch_images = batch_images.to(device)
        batch_depths = batch_depths.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.no_grad():
            feats = depth_model.encoder(batch_images)
        loss = criterion(depth_model.decode(feats), batch_depths)
        loss.backward(); optimizer.step()
        running.update(loss.item(), batch_images.shape[0])

    metrics = evaluate_depth(depth_model, val_loader, device)
    history['loss'].append(running.avg)
    history['val_rmse'].append(metrics['rmse'])
    print(f'epoch {epoch}  loss {running.avg:.4f}')
    print(format_metrics(metrics, prefix='  val: '))

## Phase 4 — Validate and visualise (Role 3)

In [ ]:
from IPython.display import Image as IPyImage, display
from validation.visualize import plot_curves, visualize_depth_model

path = visualize_depth_model(depth_model, val_loader, device,
                             os.path.join(config.RESULTS_DIR, 'nb_depth.png'),
                             num_samples=3, title='Depth — RGB | GT | SNN')
display(IPyImage(filename=path))

path = plot_curves(history, os.path.join(config.RESULTS_DIR, 'nb_curves.png'))
display(IPyImage(filename=path))

## Track B — SSD detection on the spiking trunk

Same four phases. Only the MobileNet stages are converted and frozen; the SSD
extra layers and both heads stay continuous.

In [ ]:
from data.loaders import build_detection_loaders, class_names_of, num_classes_of
from losses.multibox_loss import MultiBoxLoss
from pipeline import build_ssd_pipeline
from train_ssd import forward_split
from validation.metrics_detection import evaluate_detection

det_train, det_val = build_detection_loaders(
    root=config.COCO_ROOT, synthetic=SYNTHETIC, batch_size=8,
    num_workers=0, device=device, max_train=256, max_val=64)

n_classes = num_classes_of(det_train.dataset)
names = class_names_of(det_train.dataset)

ssd_model, _ = build_ssd_pipeline(det_train, device, num_classes=n_classes,
                                  pretrained=True, calibration_batches=5,
                                  lambda_=1.0, fire_fn='binary', timesteps=1)

In [ ]:
ssd_criterion = MultiBoxLoss(ssd_model.priors, num_classes=n_classes).to(device)
ssd_optimizer = torch.optim.AdamW(ssd_model.trainable_parameters(), lr=1e-3,
                                  weight_decay=config.WEIGHT_DECAY)
ssd_history = {'loss': [], 'mAP': []}

for epoch in range(1, 16):
    ssd_model.train()
    running = utils.AverageMeter()
    for batch_images, boxes, labels in det_train:
        batch_images = batch_images.to(device)
        ssd_optimizer.zero_grad(set_to_none=True)
        loc, scores = forward_split(ssd_model, batch_images, timesteps=1)
        loss = ssd_criterion(loc, scores, boxes, labels)
        loss.backward(); ssd_optimizer.step()
        running.update(loss.item(), batch_images.shape[0])

    metrics = evaluate_detection(ssd_model, det_val, device,
                                 num_classes=n_classes, class_names=names)
    ssd_history['loss'].append(running.avg)
    ssd_history['mAP'].append(metrics['mAP'])
    print(f'epoch {epoch}  loss {running.avg:.4f}  mAP@0.5 {metrics["mAP"]:.4f}')

In [ ]:
from validation.visualize import visualize_detection_model

path = visualize_detection_model(
    ssd_model, det_val, device,
    os.path.join(config.RESULTS_DIR, 'nb_detect.png'),
    num_samples=4, class_names=names, score_threshold=0.3,
    title='Detection — solid = predicted, dashed = ground truth')
display(IPyImage(filename=path))

path = plot_curves(ssd_history, os.path.join(config.RESULTS_DIR, 'nb_ssd_curves.png'))
display(IPyImage(filename=path))

## Ablation — λ vs spike rate vs accuracy

λ scales every threshold together. Lower λ fires more neurons (more energy) but
at T=1 with a *binary* neuron it also saturates more of them, collapsing
magnitude information. Pair a low λ with `fire_fn='mtn'` if you want the
paper's operating point.

In [ ]:
from calibration.lambda_search import measure_spike_rate
from models.snn import set_lambda
from validation.metrics_depth import evaluate_depth_rmse

rows = []
for lam in config.LAMBDA_SEARCH_GRID:
    set_lambda(depth_model.encoder, lam)
    rmse = evaluate_depth_rmse(depth_model, val_loader, device)
    rate = measure_spike_rate(depth_model.encoder, val_loader, device, 2)['overall']
    rows.append((lam, rate, rmse))
    print(f'lambda {lam:<5}  spike_rate {rate:.3f}  val RMSE {rmse:.4f} m')
set_lambda(depth_model.encoder, 1.0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([r[0] for r in rows], [r[2] for r in rows], 'o-', color='#E45756', label='val RMSE (m)')
ax.set_xlabel('lambda'); ax.set_ylabel('RMSE (m)')
ax2 = ax.twinx()
ax2.plot([r[0] for r in rows], [r[1] for r in rows], 's--', color='#4C78A8', label='spike rate')
ax2.set_ylabel('spike rate')
fig.legend(loc='upper center', ncol=2)
plt.tight_layout(); plt.show()